## Attribute Classification

**Information Need:** Understand the value characteristics of a selected event log attribute according to whether it represents categorical or quantitative information and, for quantitative attributes, whether its values exhibit monotonic behavior over time, as well as cumulative vs. incremental. For attributes with an incorrect datatype, cast them to the correct type.

**Motivation:** Event log attributes may represent quantitative or categorical information and therefore require different forms of descriptive analysis. Some quantitative attributes may represent quantities that increase or decrease over the course of a case, such as accumulated costs or payments. If an attribute is interpreted as cumulative, for example, its values are expected not to decrease as the case progresses. Applying type-appropriate descriptions and verifying expected monotonic behavior helps analysts develop an initial understanding of the information represented by an attribute and identify cases in which expected behavior is violated. Casting incorrect datatypes to the types correctly representative of their values facilitates automated analyses.

**Preconditions:** If casting is required, the target datatype is specified.

**Approach:** Characterize a selected attribute according to whether it represents categorical or quantitative information. For quantitative attributes, where relevant, examine the temporal evolution of the attribute within each case and assess whether successive observed values are monotonically increasing or decreasing.

**Output:** The classification of the selected attribute as categorical or quantitative; for quantitative attributes an assessment whether they are cumulative or incremental; for quantitative attributes for which monotonicity is assessed, an assessment of whether the attribute exhibits monotonic behavior within cases, together with the cases that violate this behavior. If casting is applied, the log represents the respective attributes with the target datatype.

In [ ]:
import pandas as pd
import pm4py

from ipywidgets import interact


# --- Configuration -----------------------------------------------------------
LOG_PATH = "../../data/Road_Traffic_Fine_Management_Process.xes"

CASE_ID = "case:concept:name"
ACTIVITY = "concept:name"
TIMESTAMP = "time:timestamp"

MANDATORY_COLUMNS = {CASE_ID, ACTIVITY, TIMESTAMP}

In [ ]:
event_log = pm4py.read_xes(LOG_PATH)

display(event_log.head())

### Pattern execution

#### 1. Identification and characterization of categorical attributes

Candidate attributes are all attributes that are non-mandatory and non-numeric

In [ ]:
candidate_categorical_attributes = [
    c for c in event_log.columns
    if c not in MANDATORY_COLUMNS and not pd.api.types.is_numeric_dtype(event_log[c])
]

pd.DataFrame({
    'attribute': candidate_categorical_attributes,
    'dtype': [event_log[c].dtype for c in candidate_categorical_attributes],
})

In [ ]:
def value_distribution(attribute):
    # dropna=False keeps missing values in the counts instead of dropping them
    counts = event_log[attribute].value_counts(dropna=False)
    counts.index = counts.index.map(lambda value: 'missing' if pd.isna(value) else value)

    percent = (counts / len(event_log) * 100).round(2)
    
    return pd.DataFrame({
        'count': counts.astype(int),
        'percent': percent
    })


@interact(attribute=candidate_categorical_attributes)
def show_value_distribution(attribute):
    return value_distribution(attribute)

#### 2. Identification and characterization of quantiative attributes

Candidate attributes are all attributes that are non-mandatory and numeric

In [ ]:
candidate_numeric_attributes = [
    c for c in event_log.columns
    if c not in MANDATORY_COLUMNS and pd.api.types.is_numeric_dtype(event_log[c])
]

pd.DataFrame({
    'attribute': candidate_numeric_attributes,
    'dtype': [event_log[c].dtype for c in candidate_numeric_attributes],
})

In [ ]:
def numeric_summary(attribute):
    values = event_log[attribute].dropna()
    return pd.Series({
        'average': values.mean(),
        'variance': values.var(),
        'stdev': values.std(),
        'median': values.median(),
    }, name=attribute)


@interact(attribute=candidate_numeric_attributes)
def show_numeric_summary(attribute):
    return numeric_summary(attribute)

In [ ]:
# sorting is independent of the chosen attribute, so it is done once upfront
by_case = event_log.sort_values([CASE_ID, TIMESTAMP])
by_log = event_log.sort_values(TIMESTAMP)

def case_level_monotonic_flags(attribute):
    def is_monotonic(series):
        return series.dropna().is_monotonic_increasing
    return by_case.groupby(CASE_ID)[attribute].apply(is_monotonic)


def is_log_level_monotonic(attribute):
    return by_log[attribute].dropna().is_monotonic_increasing

In [ ]:
@interact(attribute=candidate_numeric_attributes)
def verify_cumulative_attribute(attribute):
    case_flags = case_level_monotonic_flags(attribute)
    case_level = case_flags.all()
    log_level = is_log_level_monotonic(attribute)
    print(f'{attribute} monotonically increasing per case: {case_level}')
    print(f'{attribute} monotonically increasing over the whole log: {log_level}')

    violating_cases = case_flags[~case_flags].index.tolist()
    print(f'Cases where {attribute} is not monotonically increasing ({len(violating_cases)}): {violating_cases}')

In [ ]:
def case_level_monotonic_decreasing_flags(attribute):
    def is_monotonic(series):
        return series.dropna().is_monotonic_decreasing
    return by_case.groupby(CASE_ID)[attribute].apply(is_monotonic)


def is_log_level_monotonic_decreasing(attribute):
    return by_log[attribute].dropna().is_monotonic_decreasing

In [ ]:
@interact(attribute=candidate_numeric_attributes)
def verify_decreasing_attribute(attribute):
    case_flags = case_level_monotonic_decreasing_flags(attribute)
    case_level = case_flags.all()
    log_level = is_log_level_monotonic_decreasing(attribute)
    print(f'{attribute} monotonically decreasing per case: {case_level}')
    print(f'{attribute} monotonically decreasing over the whole log: {log_level}')

    violating_cases = case_flags[~case_flags].index.tolist()
    print(f'Cases where {attribute} is not monotonically decreasing ({len(violating_cases)}): {violating_cases}')